# ARTICLE2_NB1: Reference reproduction and seed runs

**This notebook produces results for the fault-injection study (see README).**

Protocol: *study protocol*, Section 2 (reference pipeline and reproduction check).

**What it does**
- **Part A (exact check, no training):** recomputes the first article's Table 4 and the primary Diebold-Mariano results from the saved v2 forecast files. Any mismatch stops the notebook.
- **Part B (retrain check, seed 42):** retrains LSTM and XGBoost (base and serial) with the original seed and rebuilds all architectures. Compares with the saved v2 forecasts. A small difference is possible on a different Colab GPU; it is recorded, not hidden.
- **Part C (five-seed reference, runs only after the protocol is frozen):** repeats Part B for seeds S0-S4 and saves the reference seed range that every fault will be judged against.

**Inputs and outputs:** everything for this study lives in `My Drive/Article 2`; nothing outside that folder is read or written.
**Inputs (read only):** `Article 2/02_Inputs_v2_snapshot` (a frozen copy of the verified v2 files, taken 11 Nov 2026; a checksum manifest stops the notebook if any file changes).
**Outputs:** `Article 2/05_Outputs` only.
**Code:** `Article 2/03_Code/article2_pipeline.py`. Use a GPU runtime (T4 if available, as in v2).

## CELL 0: Environment setup (Article 2 folder only)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

%cd "/content/drive/MyDrive/Article 2"

import os, sys
from pathlib import Path
ROOT = Path(os.getcwd())
assert ROOT.name == "Article 2", f"Wrong folder: {ROOT}"
CODE_DIR = ROOT / "03_Code"
assert (CODE_DIR / "article2_pipeline.py").exists(), f"article2_pipeline.py missing from {CODE_DIR}"
sys.path.insert(0, str(CODE_DIR))
print("Working directory:", ROOT)

## CELL 1: Imports, versions and settings

In [ ]:
import json, time, importlib, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
import sklearn, scipy
warnings.filterwarnings("ignore")

import article2_pipeline as ap
importlib.reload(ap)

P = ap.paths(ROOT)
OUT = P["OUT"]
for d in [OUT / "reference", OUT / "logs"]:
    d.mkdir(parents=True, exist_ok=True)

# Seeds (Protocol Section 2). S0 = 42 is the seed of the published v2 run.
SEEDS = {"S0": 42, "S1": 101, "S2": 202, "S3": 303, "S4": 404}
# Set to True only after the protocol is frozen and the seeds above match it (Protocol Section 8).
PROTOCOL_FROZEN = False

VERSIONS = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
            "tensorflow": tf.__version__, "xgboost": xgb.__version__, "scikit-learn": sklearn.__version__,
            "scipy": scipy.__version__, "gpu": [d.name for d in tf.config.list_physical_devices("GPU")],
            "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
for k, v in VERSIONS.items():
    print(f"{k:14s}: {v}")
print("Seeds:", SEEDS, "| protocol frozen:", PROTOCOL_FROZEN)
print("Input snapshot:", ap.check_input_snapshot(P))

## CELL 2: Part A - exact check against the first article (no training)

Published values are typed in from the accepted Forecasting article (Table 3 persistence MAE, Table 4 MAE ratios, Table 6 primary DM tests). The saved v2 forecasts must reproduce them exactly.

In [ ]:
PUBLISHED_PERSISTENCE_MAE = {"JPM": 1.5696, "NEE": 0.7757, "XOM": 1.0875}
PUBLISHED_TABLE4 = pd.DataFrame({
    "JPM": [0.993, 1.000, 1.047, 1.026, 1.002, 1.002, 1.000, 1.055],
    "NEE": [1.002, 1.001, 1.008, 1.019, 1.001, 1.001, 1.002, 1.025],
    "XOM": [1.000, 1.002, 1.037, 1.026, 1.007, 1.007, 1.002, 1.000],
}, index=ap.TABLE4_MODELS)
PUBLISHED_DM = {("JPM", "LSTM", "Stacking"): 0.0027, ("JPM", "Parallel InvError", "Stacking"): 0.0013}

saved = {t: pd.read_csv(P["COMBINED"] / f"{t}_test_all_models.csv", parse_dates=["Date"], index_col="Date")
         for t in ap.TICKERS}

checkA = {}
for t in ap.TICKERS:
    pm = ap.mean_absolute_error(saved[t]["y_true"], saved[t]["Persistence"])
    checkA[f"{t}_rows_501"] = len(saved[t]) == 501
    checkA[f"{t}_persistence_MAE"] = abs(pm - PUBLISHED_PERSISTENCE_MAE[t]) <= 0.00005 + 1e-12

ratios_saved = ap.mae_ratio_table(saved)
diff = (ratios_saved.loc[ap.TABLE4_MODELS, ap.TICKERS] - PUBLISHED_TABLE4).abs()
checkA["table4_matches_3dp"] = bool((diff <= 0.0005 + 1e-9).all().all())

dm_saved = ap.primary_dm(saved)
sig_set = {(r.Ticker, r.Model_A, r.Model_B) for r in dm_saved.itertuples() if r.Sig_Holm}
checkA["dm_significant_set_matches"] = sig_set == set(PUBLISHED_DM)
for key, p in PUBLISHED_DM.items():
    row = dm_saved[(dm_saved.Ticker == key[0]) & (dm_saved.Model_A == key[1]) & (dm_saved.Model_B == key[2])]
    checkA[f"dm_p_{key[0]}_{key[1]}_vs_{key[2]}"] = abs(float(row["p_HLN"].iloc[0]) - p) <= 0.00005 + 1e-12

print("MAE ratio to persistence from saved v2 forecasts:")
print(ratios_saved.loc[ap.TABLE4_MODELS].round(3).to_string())
print()
print(pd.Series(checkA).to_string())
failedA = [k for k, v in checkA.items() if not v]
assert not failedA, f"PART A FAILED: {failedA}. Stop and investigate before any other run."
print()
print("Part A: ALL PASSED (saved v2 forecasts reproduce the published numbers)")

## CELL 3: Part B - retrain check with seed 42 (about 3-6 minutes on a T4)

Retrains every learned component with the original seed. If the result matches the saved forecasts to three decimals in every MAE ratio, the reference is fully reproducible in this environment. If not, the difference is recorded and **all fault comparisons use references re-run in this same environment** (Part C), so the environment cannot be mistaken for a fault effect.

In [ ]:
data = ap.load_data(P)
arfima = ap.load_arfima(P)

t0 = time.time()
rerun42, logs42 = ap.run_reference(P, data, arfima, SEEDS["S0"])
secs42 = time.time() - t0

ratios_rerun = ap.mae_ratio_table(rerun42)
ratio_diff = (ratios_rerun - ratios_saved).loc[ratios_saved.index, ap.TICKERS]
max_usd = {t: float((rerun42[t].drop(columns=["y_true", "Persistence"])
                     - saved[t][rerun42[t].columns.drop(["y_true", "Persistence"])]).abs().max().max())
           for t in ap.TICKERS}

print(f"Retrain time: {secs42:.0f} s")
print("Largest absolute forecast difference vs saved v2 (USD):", {k: round(v, 6) for k, v in max_usd.items()})
print()
print("MAE ratio difference (rerun minus saved):")
print(ratio_diff.round(4).to_string())
print()
print("Training logs (seed 42):")
print(logs42["base"].to_string(index=False))
print(logs42["serial"].to_string(index=False))

max_ratio_diff = float(ratio_diff.abs().max().max())
if max(max_usd.values()) < 1e-6:
    VERDICT_B = "EXACT"
elif max_ratio_diff <= 0.0005:
    VERDICT_B = "MATCHES_3DP"
else:
    VERDICT_B = "ENVIRONMENT_DRIFT"
print()
print("Part B verdict:", VERDICT_B, f"(largest MAE ratio difference {max_ratio_diff:.4f})")
if VERDICT_B == "ENVIRONMENT_DRIFT":
    print("Record this in the protocol deviation log. Fault comparisons will use Part C references only.")

## CELL 4: Part C - five-seed reference (runs only after the protocol is frozen)

Each seed retrains LSTM and XGBoost (base and serial) and rebuilds every architecture. ARFIMA is deterministic and is not retrained. Expect roughly 15-30 minutes in total on a T4.

In [ ]:
assert PROTOCOL_FROZEN, "Protocol not frozen yet. Freeze it (Protocol Section 8), confirm SEEDS, then set PROTOCOL_FROZEN = True."

ref_ratios, ref_dm, all_logs = [], [], {"base": [], "serial": [], "weights": [], "stacking": []}
for label, seed in SEEDS.items():
    t0 = time.time()
    comb, logs = (rerun42, logs42) if label == "S0" else ap.run_reference(P, data, arfima, seed)
    sdir = OUT / "reference" / f"seed_{seed}"
    sdir.mkdir(parents=True, exist_ok=True)
    for t in ap.TICKERS:
        comb[t].to_csv(sdir / f"{t}_test_all_models.csv")
    r = ap.mae_ratio_table(comb).loc[ap.TABLE4_MODELS + ["Serial (ARFIMA+LSTM)"]]
    r = r.reset_index().rename(columns={"index": "Model"}).assign(Seed=seed)
    ref_ratios.append(r)
    d = ap.primary_dm(comb).assign(Seed=seed)
    ref_dm.append(d)
    for k in all_logs:
        all_logs[k].append(logs[k])
    print(f"{label} (seed {seed}) done in {time.time() - t0:.0f} s | significant primary DM after Holm: {int(d.Sig_Holm.sum())}")

ref_ratios = pd.concat(ref_ratios, ignore_index=True)
ref_dm = pd.concat(ref_dm, ignore_index=True)
ref_ratios.to_csv(OUT / "reference" / "reference_mae_ratios_by_seed.csv", index=False)
ref_dm.to_csv(OUT / "reference" / "reference_primary_dm_by_seed.csv", index=False)
for k, frames in all_logs.items():
    pd.concat(frames, ignore_index=True).to_csv(OUT / "reference" / f"reference_{k}_logs_by_seed.csv", index=False)

long = ref_ratios.melt(id_vars=["Model", "Seed"], value_vars=ap.TICKERS + ["Mean"], var_name="Stock", value_name="MAE_ratio")
seed_range = long.groupby(["Model", "Stock"])["MAE_ratio"].agg(["mean", "min", "max"])
seed_range["range"] = seed_range["max"] - seed_range["min"]
seed_range.to_csv(OUT / "reference" / "reference_seed_range.csv")
print()
print("Reference seed range (MAE ratio to persistence) - the yardstick for every fault:")
print(seed_range.round(4).to_string())

## CELL 5: CHECKPOINT and run log

In [ ]:
checks = {"part_A_passed": not failedA, "part_B_ran": "VERDICT_B" in globals()}
if PROTOCOL_FROZEN and "seed_range" in globals():
    checks["five_seeds_done"] = ref_ratios["Seed"].nunique() == 5
    checks["serial_stage2_differs_from_LSTM"] = all(
        not np.allclose(pd.read_csv(OUT / "reference" / f"seed_{s}" / f"{t}_test_all_models.csv")["Serial (ARFIMA+LSTM)"],
                        pd.read_csv(OUT / "reference" / f"seed_{s}" / f"{t}_test_all_models.csv")["LSTM"])
        for s in SEEDS.values() for t in ap.TICKERS)
print(pd.Series(checks).to_string())

log = {"notebook": "ARTICLE2_NB1_Reference_Seeds", "purpose": "Article 2 only", "root": str(ROOT),
       "run_at": datetime.now().isoformat(timespec="seconds"), "versions": VERSIONS,
       "seeds": SEEDS, "protocol_frozen": PROTOCOL_FROZEN, "part_B_verdict": globals().get("VERDICT_B"),
       "part_B_max_ratio_diff": globals().get("max_ratio_diff"), "checks": checks}
with open(OUT / "logs" / "ARTICLE2_NB1_run_log.json", "w") as f:
    json.dump(log, f, indent=2, default=str)

assert all(checks.values()), f"CHECKPOINT FAILED: {[k for k, v in checks.items() if not v]}"
print()
print("Hard checks: ALL PASSED")
print("Please share the outputs of CELLS 1, 2, 3 (and 4-5 once the protocol is frozen).")